# Notebook 03 - Agent Tools

**Project:** IncidentIQ - AI-powered Incident Intelligence

**Goal:** Build all LangChain tools that the LangGraph agent will use in Notebook 04.
Each tool is a self-contained function the agent can call autonomously based on user intent.

## What this notebook covers
1. Install and import all required packages
2. Tool 1 - YouTube transcript tool
3. Tool 2 - RAG knowledge search tool
4. Tool 3 - Video summarization tool
5. Tool 4 - PDF cheatsheet generator tool
6. Tool 5 - Gmail sender tool
7. Tool 6 - Related YouTube videos finder
8. Collect all tools
9. Run quality tests

## Why this matters
Tools are what make an LLM an agent. Without tools the agent can only answer questions.
With tools the agent can fetch videos, search knowledge, generate PDFs, send emails
and find related training content autonomously.
The LangGraph router in Notebook 04 decides which tool to call based on user intent.

## Who can use IncidentIQ
IncidentIQ is designed for any emergency or security organization.
- Fire services
- Police forces
- Emergency medical services
- Civil protection
- Any incident training context

---

## 1. Install required packages

Installing all dependencies needed for the six agent tools.
Run this cell once - skip on subsequent runs.

In [1]:
!pip install langchain langchain-openai langchain-community chromadb youtube-transcript-api reportlab google-api-python-client google-auth-oauthlib tavily-python python-dotenv -q
print('Packages installed.')


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Packages installed.


## 2. Import libraries and load environment variables

We import all libraries needed across the six tools.
LangChain's @tool decorator transforms regular Python functions into agent-callable tools.

In [3]:
import os
import re
import base64
import json
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

from langchain.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate

from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib.colors import HexColor, white
from reportlab.pdfgen import canvas

from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

from tavily import TavilyClient
import chromadb

load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not found. Check your .env file.'
assert os.getenv('TAVILY_API_KEY'), 'TAVILY_API_KEY not found. Check your .env file.'
print('Environment variables loaded successfully.')

Environment variables loaded successfully.


## 3. Configuration

Central configuration shared across all tools.
Must match the values used in Notebook 01 and 02.

In [4]:
# Must match Notebook 01 and 02
CHROMA_PATH      = '../chroma_db'
COLLECTION_NAME  = 'incidentiq'
EMBEDDING_MODEL  = 'text-embedding-3-small'
LLM_MODEL        = 'gpt-4o-mini'
RETRIEVER_K      = 8

# Brand colors for PDF cheatsheet
RED    = HexColor('#C0392B')   # IncidentIQ red
DARK   = HexColor('#1C2833')   # Dark grey
ORANGE = HexColor('#E67E22')   # Accent orange
GREEN  = HexColor('#1E8449')   # Recommendations green
WHITE  = white

# Gmail distribution list - loaded from .env for security
# Format in .env: GMAIL_DISTRIBUTION_LIST=email1@domain.be,email2@domain.be
DISTRIBUTION_LIST = os.getenv('GMAIL_DISTRIBUTION_LIST', '').split(',')

# Gmail OAuth scopes
GMAIL_SCOPES = ['https://www.googleapis.com/auth/gmail.send']

# Initialize shared LLM, embeddings, vectorstore and web search client
llm             = ChatOpenAI(model=LLM_MODEL, temperature=0)
embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)
chroma_client   = chromadb.PersistentClient(path=CHROMA_PATH)
vectorstore     = Chroma(
    client=chroma_client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
)
tavily = TavilyClient(api_key=os.getenv('TAVILY_API_KEY'))

print('Configuration set.')
print(f'   LLM model:          {LLM_MODEL}')
print(f'   Embedding model:    {EMBEDDING_MODEL}')
print(f'   ChromaDB path:      {CHROMA_PATH}')
print(f'   Distribution list:  {DISTRIBUTION_LIST}')

Configuration set.
   LLM model:          gpt-4o-mini
   Embedding model:    text-embedding-3-small
   ChromaDB path:      ../chroma_db
   Distribution list:  ['']


/var/folders/9n/zs24nr4n1jq9k_zjkcrvfj1h0000gn/T/ipykernel_23615/2926752329.py:26: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectorstore     = Chroma(


## 4. Tool 1 - YouTube transcript tool

This tool fetches the transcript of a YouTube video and stores it in ChromaDB.
The agent calls this tool automatically when it detects a YouTube URL in the user message.

What it does:
1. Extracts the video ID from the URL
2. Fetches the official subtitles via youtube-transcript-api
3. Cleans noise tags like [Music] and empty timestamps
4. Splits the transcript into chunks
5. Embeds and stores chunks in ChromaDB
6. Returns a confirmation message to the agent

Supported content: Any YouTube video with subtitles enabled - incident debriefings,
tactical presentations, procedure walkthroughs, training recordings.

In [5]:
def extract_video_id(url: str) -> str:
    """Extract YouTube video ID from standard (watch?v=) or shortened (youtu.be/) URL."""
    if 'v=' in url:
        return url.split('v=')[1].split('&')[0]
    elif 'youtu.be/' in url:
        return url.split('youtu.be/')[1].split('?')[0]
    raise ValueError(f'Cannot extract video ID from URL: {url}')


def clean_transcript(text: str) -> str:
    """
    Remove noise tags and empty timestamps from YouTube transcripts.
    Tags like [Music] and empty timestamps add no informational value
    and reduce retrieval quality in ChromaDB.
    """
    text = re.sub(r'\[Music\]|\[Applause\]|\[Laughter\]|\[Cheering\]', '', text)
    text = re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


@tool
def fetch_youtube_transcript(youtube_url: str) -> str:
    """
    Fetch the transcript of a YouTube video and store it in ChromaDB.
    Use this tool when the user provides a YouTube URL.
    Supports English, Dutch and French transcripts.
    Works with any incident training video that has subtitles enabled.
    Returns a confirmation with the number of chunks stored.
    """
    try:
        video_id = extract_video_id(youtube_url)
        try:
            entries = YouTubeTranscriptApi().fetch(video_id, languages=['en', 'nl', 'fr'])
            transcript_list = entries.snippets
        except NoTranscriptFound:
            return (
                f'No transcript found for video {video_id}. '
                'Please use a video with subtitles enabled (CC button visible in YouTube).'
            )
        except TranscriptsDisabled:
            return f'Transcripts are disabled for video {video_id}.'

        plain_text = ' '.join(t.text for t in transcript_list)
        timestamped_text = ' '.join(
            f'[{int(t.start//60):02d}:{int(t.start%60):02d}] {t.text}'
            for t in transcript_list
        )
        plain_text       = clean_transcript(plain_text)
        timestamped_text = clean_transcript(timestamped_text)

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=50, separators=['\n\n', '\n', '. ', ' ']
        )
        chunks = splitter.create_documents(
            texts=[timestamped_text],
            metadatas=[{'video_id': video_id, 'source': youtube_url}]
        )

        existing = vectorstore.get(where={'video_id': video_id})
        if existing['ids']:
            vectorstore.delete(ids=existing['ids'])
        vectorstore.add_documents(chunks)

        return (
            f'Transcript loaded successfully.\n'
            f'Video ID: {video_id}\n'
            f'Total characters: {len(plain_text):,}\n'
            f'Chunks stored in ChromaDB: {len(chunks)}\n'
            f'Ready for Q&A.'
        )
    except Exception as e:
        return f'Error fetching transcript: {str(e)}'


print('Testing Tool 1 - fetch_youtube_transcript...')
result = fetch_youtube_transcript.invoke({'youtube_url': 'https://www.youtube.com/watch?v=7OH5FEWWM_k'})
print(result)

Testing Tool 1 - fetch_youtube_transcript...
Transcript loaded successfully.
Video ID: 7OH5FEWWM_k
Total characters: 19,793
Chunks stored in ChromaDB: 54
Ready for Q&A.


## 5. Tool 2 - RAG knowledge search tool

This tool searches the ChromaDB knowledge base for relevant information.
The agent calls this tool to answer any question about the loaded video content.

What it does:
1. Translates the query to English for better retrieval
2. Searches ChromaDB using cosine similarity
3. Cleans retrieved chunks and extracts timestamps
4. Returns relevant context with source citations to the agent

Why query translation matters:
ChromaDB contains English transcript chunks. Searching in Dutch or French
produces less relevant results. Translating to English first ensures
consistent retrieval quality regardless of the user language.

In [6]:
@tool
def search_video_knowledge(query: str) -> str:
    """
    Search the ChromaDB knowledge base for information relevant to the query.
    Use this tool to answer questions about the loaded video content.
    Automatically translates non-English queries for better retrieval quality.
    Works with questions in any language about any incident training video.
    Returns the most relevant transcript excerpts with timestamp sources.
    """
    try:
        english_query = llm.invoke(
            f'Translate to English, return only the translation: {query}'
        ).content.strip()

        results = vectorstore.similarity_search(english_query, k=RETRIEVER_K)

        if not results:
            return 'No relevant information found. Please load a YouTube video first.'

        all_timestamps = re.findall(
            r'\[\d{2}:\d{2}\]',
            ' '.join([r.page_content for r in results])
        )
        seen = set()
        unique_timestamps = []
        for t in all_timestamps:
            if t not in seen:
                seen.add(t)
                unique_timestamps.append(t)

        clean_chunks = [
            re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', r.page_content)
            for r in results
        ]
        context = '\n\n'.join(clean_chunks)
        sources = ' | '.join(unique_timestamps[:5])

        return f'{context}\n\nSources: {sources}'

    except Exception as e:
        return f'Error searching knowledge base: {str(e)}'


print('Testing Tool 2 - search_video_knowledge...')
result = search_video_knowledge.invoke({'query': 'Welke fouten werden gemaakt tijdens het incident?'})
print(result[:500] + '...')

Testing Tool 2 - search_video_knowledge...
[22:24] themselves so different stations they [22:26] were not collaborating very well [22:28] everybody wanted to extinguish the fire [22:30] it's my fire you know uh and all that [22:32] changed and he said it's thanks to all [22:35] these changes that even with all the [22:37] problems we account of today this became [22:40] a success because nobody died nobody got [22:42] hurt in a way no firefighter got hurt it [22:45] was tough but we managed to do it and we [22:47] we drew some lessons

t...


## 6. Tool 3 - Video summarization tool

This tool generates a structured summary of the entire loaded video.
The agent calls this tool when the user asks for a full summary or overview.

What it does:
1. Retrieves a broad set of chunks covering the full video
2. Generates a structured summary using the LLM
3. Returns the summary in the user language

Output structure:
- Introduction - what the video covers
- Key Points - most important facts
- Lessons Learned - actionable takeaways
- Conclusion - main message

In [7]:
@tool
def summarize_video(language: str = 'english') -> str:
    """
    Generate a structured summary of the entire loaded video.
    Use this tool when the user asks for a full summary or overview of the video.
    Specify language as 'english', 'dutch' or 'french'.
    Works with any incident training video loaded into ChromaDB.
    Returns a structured summary with introduction, key points, lessons and conclusion.
    """
    try:
        results = vectorstore.similarity_search(
            'main topic lessons learned conclusions key points', k=12
        )
        if not results:
            return 'No video content found. Please load a YouTube video first.'

        context = '\n\n'.join([
            re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', r.page_content)
            for r in results
        ])

        lang_map = {
            'english': 'English',
            'dutch':   'Dutch - use natural direct language as used in Belgian incident training',
            'french':  'French',
        }
        lang_instruction = lang_map.get(language.lower(), 'English')

        summary_prompt = (
            f'You are an incident training expert summarizing a training video.\n'
            f'Write a structured summary in {lang_instruction} based on the context below.\n\n'
            f'Use this exact structure:\n'
            f'**Introduction**\n'
            f'- [What the video is about - 2 bullets max]\n\n'
            f'**Key Points**\n'
            f'- [Most important facts - 4 to 6 bullets]\n\n'
            f'**Lessons Learned**\n'
            f'- [Actionable lessons - 3 to 5 bullets]\n\n'
            f'**Conclusion**\n'
            f'- [Main takeaway - 1 to 2 bullets]\n\n'
            f'Rules:\n'
            f'- Keep every bullet short - maximum 15 words\n'
            f'- Use simple direct language\n'
            f'- Base your summary strictly on the context provided\n\n'
            f'Context:\n{context}\n\nSummary:'
        )

        response = llm.invoke(summary_prompt)
        return response.content.strip()

    except Exception as e:
        return f'Error generating summary: {str(e)}'


print('Testing Tool 3 - summarize_video...')
result = summarize_video.invoke({'language': 'dutch'})
print(result)

Testing Tool 3 - summarize_video...
**Inleiding**
- Deze video gaat over lessen geleerd van een brandinterventie.
- Het benadrukt het belang van samenwerking en training.

**Belangrijkste punten**
- Mobiele risers moeten in een rookvrije omgeving worden gebruikt.
- Slechte samenwerking tussen verschillende brandweerkorpsen leidde tot problemen.
- Er zijn nieuwe systemen voor incidentcommandocontrole geïmplementeerd.
- Het gebruik van thermische camera's was cruciaal voor situatieweten.
- Communicatie via radio was moeilijk, persoonlijke rapportage was noodzakelijk.

**Lessen geleerd**
- Train brandweermensen beter in rookbeheer en samenwerking.
- Zorg voor een rookvrije zone bij brandinterventies.
- Gebruik altijd thermische camera's voor betere situatieweten.
- Verbeter de communicatie tussen verschillende eenheden tijdens incidenten.

**Conclusie**
- Samenwerking en goede training zijn essentieel voor succesvolle brandbestrijding.
- Lessen uit deze interventie moeten worden toegepast

## 7. Tool 4 - PDF cheatsheet generator tool

This tool generates a professional 1-page PDF cheatsheet from the video content.
The agent calls this tool when the user asks for a cheatsheet or key concepts document.

What it does:
1. Retrieves relevant chunks from ChromaDB
2. Extracts structured keypoints and recommendations using the LLM
3. Generates a branded 1-page PDF using ReportLab
4. Saves the PDF to a temporary file
5. Returns the file path for the Gmail tool to attach

Design: Branded IncidentIQ layout with clear sections, color coding
and an AI disclaimer on the recommendations section.

In [8]:
def extract_keypoints_for_pdf(context: str, language: str = 'dutch') -> dict:
    """
    Extract structured keypoints and recommendations from video context for the PDF.
    Uses the LLM to produce a clean JSON structure with title, keypoints and recommendations.
    """
    lang_map = {'dutch': 'Dutch', 'english': 'English', 'french': 'French'}
    lang = lang_map.get(language.lower(), 'Dutch')

    prompt = (
        f'Extract structured information from the context below for an incident training cheatsheet.\n'
        f'Respond in {lang} with this exact JSON format and nothing else:\n'
        f'{{\n'
        f'  "title": "Short descriptive title of the incident or training topic",\n'
        f'  "subtitle": "Source or presenter name if mentioned in the context",\n'
        f'  "keypoints": ["point 1", "point 2", "point 3", "point 4", "point 5"],\n'
        f'  "recommendations": ["rec 1", "rec 2", "rec 3", "rec 4"]\n'
        f'}}\n\n'
        f'Rules:\n'
        f'- Maximum 12 words per keypoint or recommendation\n'
        f'- Use simple direct language - no jargon\n'
        f'- Focus on actionable facts for incident commanders and responders\n'
        f'- Never include timestamps in the output\n\n'
        f'Context:\n{context}\n\nJSON:'
    )

    response = llm.invoke(prompt)
    raw = response.content.strip()
    raw = re.sub(r'```json|```', '', raw).strip()
    return json.loads(raw)


def generate_pdf(data: dict, source_url: str = '') -> str:
    """
    Generate a branded 1-page PDF cheatsheet using ReportLab.
    Returns the absolute file path of the generated PDF.
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath  = f'/tmp/incidentiq_cheatsheet_{timestamp}.pdf'
    c         = canvas.Canvas(filepath, pagesize=A4)
    WIDTH, HEIGHT = A4

    # Header bar
    c.setFillColor(RED)
    c.rect(0, HEIGHT - 3.2*cm, WIDTH, 3.2*cm, fill=1, stroke=0)
    c.setFillColor(WHITE)
    c.circle(1.8*cm, HEIGHT - 1.6*cm, 0.85*cm, fill=1, stroke=0)
    c.setFillColor(RED)
    c.setFont('Helvetica-Bold', 14)
    c.drawCentredString(1.8*cm, HEIGHT - 1.95*cm, 'IQ')
    c.setFillColor(WHITE)
    c.setFont('Helvetica-Bold', 15)
    c.drawString(3.2*cm, HEIGHT - 1.3*cm, data.get('title', 'IncidentIQ Cheatsheet'))
    c.setFont('Helvetica', 10)
    c.drawString(3.2*cm, HEIGHT - 1.85*cm, data.get('subtitle', ''))
    c.setFont('Helvetica', 8)
    c.drawRightString(WIDTH - 1.2*cm, HEIGHT - 1.3*cm, datetime.now().strftime('%d/%m/%Y'))
    c.drawRightString(WIDTH - 1.2*cm, HEIGHT - 1.75*cm, 'Generated by IncidentIQ AI')
    c.setFillColor(ORANGE)
    c.rect(0, HEIGHT - 3.6*cm, WIDTH, 0.4*cm, fill=1, stroke=0)

    y = HEIGHT - 5.0*cm

    def section_header(y, title, color=DARK):
        c.setFillColor(color)
        c.setFont('Helvetica-Bold', 11)
        c.drawString(1.2*cm, y, title.upper())
        c.setStrokeColor(color)
        c.setLineWidth(1.5)
        c.line(1.2*cm, y - 0.2*cm, WIDTH - 1.2*cm, y - 0.2*cm)
        return y - 0.7*cm

    def bullet_item(y, text, color=DARK, bullet_color=RED):
        c.setFillColor(bullet_color)
        c.circle(1.5*cm - 0.3*cm, y + 0.2*cm, 0.1*cm, fill=1, stroke=0)
        c.setFillColor(color)
        c.setFont('Helvetica', 9.5)
        max_w = WIDTH - 1.5*cm - 1.2*cm
        words = text.split()
        line, lines = '', []
        for word in words:
            test = line + word + ' '
            if c.stringWidth(test, 'Helvetica', 9.5) < max_w:
                line = test
            else:
                lines.append(line.strip())
                line = word + ' '
        lines.append(line.strip())
        for i, l in enumerate(lines):
            c.drawString(1.5*cm, y - i * 0.45*cm, l)
        return y - len(lines) * 0.45*cm - 0.3*cm

    y = section_header(y, 'Key Points', RED)
    for kp in data.get('keypoints', []):
        y = bullet_item(y, kp)
    y -= 0.4*cm

    y = section_header(y, 'AI-generated recommendations based on the video', GREEN)
    c.setFillColor(HexColor('#EAFAF1'))
    c.setStrokeColor(GREEN)
    c.setLineWidth(0.8)
    c.roundRect(1.2*cm, y - 0.75*cm, WIDTH - 2.4*cm, 0.65*cm, 3, fill=1, stroke=1)
    c.setFillColor(GREEN)
    c.setFont('Helvetica-Bold', 8)
    c.drawString(1.55*cm, y - 0.3*cm, 'AI analysis:')
    c.setFillColor(DARK)
    c.setFont('Helvetica-Oblique', 8)
    c.drawString(3.0*cm, y - 0.3*cm,
        'Recommendations below are automatically extracted from the video - not cited from a person.')
    y -= 1.0*cm
    for rec in data.get('recommendations', []):
        y = bullet_item(y, rec, bullet_color=GREEN)

    c.setFillColor(RED)
    c.rect(0, 1.2*cm, WIDTH, 0.15*cm, fill=1, stroke=0)
    c.setFillColor(DARK)
    c.rect(0, 0, WIDTH, 1.2*cm, fill=1, stroke=0)
    c.setFillColor(WHITE)
    c.setFont('Helvetica', 7.5)
    c.drawString(1.2*cm, 0.65*cm, 'IncidentIQ - AI-powered Incident Intelligence')
    if source_url:
        c.drawCentredString(WIDTH/2, 0.65*cm, f'Source: {source_url[:70]}')
    c.drawRightString(WIDTH - 1.2*cm, 0.65*cm, 'Page 1/1')
    c.save()
    return filepath


@tool
def generate_pdf_cheatsheet(language: str = 'dutch', source_url: str = '') -> str:
    """
    Generate a professional 1-page PDF cheatsheet from the loaded video content.
    Use this tool when the user asks for a cheatsheet, key concepts document or PDF summary.
    Specify language as 'dutch', 'english' or 'french'.
    Works with any incident training video loaded into ChromaDB.
    Returns the file path of the generated PDF for the Gmail tool to attach.
    """
    try:
        results = vectorstore.similarity_search(
            'key points lessons learned recommendations conclusions', k=10
        )
        if not results:
            return 'No video content found. Please load a YouTube video first.'

        context = '\n\n'.join([r.page_content for r in results])
        print('Extracting keypoints for PDF...')
        data = extract_keypoints_for_pdf(context, language)
        print('Generating PDF...')
        filepath = generate_pdf(data, source_url)

        return (
            f'PDF cheatsheet generated successfully.\n'
            f'File path: {filepath}\n'
            f'Title: {data.get("title", "N/A")}'
        )
    except Exception as e:
        return f'Error generating PDF: {str(e)}'


print('Testing Tool 4 - generate_pdf_cheatsheet...')
result = generate_pdf_cheatsheet.invoke({
    'language': 'dutch',
    'source_url': 'https://www.youtube.com/watch?v=7OH5FEWWM_k'
})
print(result)
pdf_path = result.split('File path: ')[1].split('\n')[0] if 'File path:' in result else None

Testing Tool 4 - generate_pdf_cheatsheet...
Extracting keypoints for PDF...
Generating PDF...
PDF cheatsheet generated successfully.
File path: /tmp/incidentiq_cheatsheet_20260520_115252.pdf
Title: Lessen geleerd van brandinterventies


## 8. Tool 5 - Gmail sender tool

This tool sends the generated PDF cheatsheet to the distribution list via Gmail API.
The agent calls this tool after the PDF cheatsheet tool has generated the file.

What it does:
1. Authenticates with Gmail via OAuth 2.0
2. Attaches the PDF to a professional email
3. Sends to the distribution list and any custom email addresses
4. Returns a confirmation with the list of recipients

One-time setup required:
- Enable Gmail API in Google Cloud Console
- Download credentials.json and place it in the project root
- First run opens a browser for OAuth authentication
- Token is saved to token.json for all subsequent runs automatically

In [9]:
def get_gmail_service():
    """
    Authenticate with Gmail API using OAuth 2.0.
    First run opens a browser for user authentication.
    Subsequent runs use saved token.json automatically.
    """
    creds      = None
    token_path = Path('../token.json')
    creds_path = Path('../credentials.json')

    assert creds_path.exists(), (
        'credentials.json not found.\n'
        'Download from Google Cloud Console > APIs > Gmail API > Credentials.\n'
        'Place credentials.json in the project root directory.'
    )

    if token_path.exists():
        creds = Credentials.from_authorized_user_file(str(token_path), GMAIL_SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(str(creds_path), GMAIL_SCOPES)
            creds = flow.run_local_server(port=0)
        token_path.write_text(creds.to_json())
    return build('gmail', 'v1', credentials=creds)


def create_email_with_attachment(sender, recipients, subject, body, pdf_path):
    """
    Create a MIME email message with PDF attachment.
    Returns the base64-encoded message ready for the Gmail API.
    """
    msg = MIMEMultipart()
    msg['From']    = sender
    msg['To']      = ', '.join(recipients)
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    with open(pdf_path, 'rb') as f:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={Path(pdf_path).name}')
    msg.attach(part)
    return {'raw': base64.urlsafe_b64encode(msg.as_bytes()).decode()}


@tool
def send_gmail_with_cheatsheet(pdf_path: str, custom_emails: str = '') -> str:
    """
    Send the generated PDF cheatsheet to the distribution list via Gmail.
    Use this tool after generate_pdf_cheatsheet has created the PDF file.
    pdf_path: the file path returned by the generate_pdf_cheatsheet tool.
    custom_emails: optional comma-separated email addresses to add to the send list.
    Works for any organization - fire service, police, EMS or civil protection.
    Returns a confirmation message with the full list of recipients.
    """
    try:
        assert Path(pdf_path).exists(), f'PDF not found at path: {pdf_path}'

        recipients = [e.strip() for e in DISTRIBUTION_LIST if e.strip()]
        if custom_emails:
            extra = [e.strip() for e in custom_emails.split(',') if e.strip()]
            recipients.extend(extra)

        assert recipients, (
            'No recipients found.\n'
            'Add emails to GMAIL_DISTRIBUTION_LIST in .env '
            'or provide custom_emails when calling this tool.'
        )

        service = get_gmail_service()
        subject = f'IncidentIQ - Key Concepts Cheatsheet - {datetime.now().strftime("%d/%m/%Y")}'
        body = (
            'Dear colleague,\n\n'
            'Please find attached the AI-generated key concepts cheatsheet '
            'from the latest incident training video.\n\n'
            'This document was automatically generated by IncidentIQ.\n'
            'AI-generated recommendations are based on video content analysis '
            'and should be reviewed by a qualified officer before operational use.\n\n'
            'Best regards,\n'
            'IncidentIQ AI Agent'
        )

        message = create_email_with_attachment('me', recipients, subject, body, pdf_path)
        service.users().messages().send(userId='me', body=message).execute()

        return (
            f'Cheatsheet sent successfully.\n'
            f'Recipients: {", ".join(recipients)}\n'
            f'Subject: {subject}'
        )
    except Exception as e:
        return f'Error sending email: {str(e)}'


print('Tool 5 - send_gmail_with_cheatsheet defined.')
print('Note: Gmail OAuth requires credentials.json in the project root.')
print('First run will open a browser for authentication.')

Tool 5 - send_gmail_with_cheatsheet defined.
Note: Gmail OAuth requires credentials.json in the project root.
First run will open a browser for authentication.


## 9. Tool 6 - Related YouTube videos finder

This tool extracts key concepts from the loaded video and searches YouTube
for related incident training videos on the same topics.
The agent calls this tool when the user asks for more resources or related content.

What it does:
1. Retrieves chunks from ChromaDB to identify the video key concepts
2. Uses the LLM to extract 3 specific technical concepts
3. Searches YouTube via Tavily for each concept
4. Returns a formatted list of related videos with titles and direct links

Why this is powerful:
One incident video contains multiple topics. This tool surfaces related
content on each topic automatically - creating a personalized learning path
from a single video drop. Works for any incident type.

In [10]:
def extract_key_concepts(context: str) -> list:
    """
    Extract the 3 main technical concepts from the video transcript.
    Used as search queries to find related YouTube videos.
    Returns a list of 3 specific search terms - maximum 4 words each.
    """
    prompt = (
        'Extract exactly 3 specific technical concepts from this incident training video context.\n'
        'Return only a JSON list of 3 short search terms - each maximum 4 words.\n'
        'Focus on specific topics, not general terms like incident or training.\n\n'
        'Example output: ["high rise fire tactics", "reverse stack effect", "mobile riser system"]\n\n'
        f'Context:\n{context}\n\nJSON list:'
    )
    response = llm.invoke(prompt)
    raw = response.content.strip()
    raw = re.sub(r'```json|```', '', raw).strip()
    return json.loads(raw)


@tool
def find_related_youtube_videos(topic_hint: str = '') -> str:
    """
    Find related YouTube incident training videos based on key concepts from the loaded video.
    Use this tool when the user asks for related videos, more resources or wants to learn more.
    topic_hint: optional specific topic to search for - if empty uses key concepts from video.
    Works for any incident type - fire, police, EMS, civil protection.
    Returns a formatted list of related YouTube videos with titles and direct links.
    """
    try:
        results = vectorstore.similarity_search(
            'main topic key concepts technical terms procedures', k=6
        )
        if not results:
            return 'No video content found. Please load a YouTube video first.'

        context = '\n\n'.join([r.page_content for r in results])

        if topic_hint:
            concepts = [f'{topic_hint} incident training']
        else:
            print('Extracting key concepts from video...')
            concepts = extract_key_concepts(context)

        print(f'Searching YouTube for: {concepts}')

        all_videos = []
        seen_urls  = set()

        for concept in concepts:
            query          = f'site:youtube.com incident training {concept}'
            search_results = tavily.search(query=query, max_results=3, search_depth='basic')

            for r in search_results.get('results', []):
                url   = r.get('url', '')
                title = r.get('title', '')
                if 'youtube.com/watch' in url and url not in seen_urls:
                    seen_urls.add(url)
                    all_videos.append({'title': title, 'url': url, 'concept': concept})

        if not all_videos:
            return 'No related YouTube videos found for the key concepts in this video.'

        output_lines    = ['**Related Incident Training Videos**\n']
        current_concept = None

        for video in all_videos[:6]:
            if video['concept'] != current_concept:
                current_concept = video['concept']
                output_lines.append(f"\n Key concept: {current_concept.title()}")
            output_lines.append(f"  - {video['title']}\n    {video['url']}")

        output_lines.append('\nBased on key concepts automatically extracted from the loaded video.')
        return '\n'.join(output_lines)

    except Exception as e:
        return f'Error finding related videos: {str(e)}'


print('Testing Tool 6 - find_related_youtube_videos...')
result = find_related_youtube_videos.invoke({'topic_hint': ''})
print(result)

Testing Tool 6 - find_related_youtube_videos...
Extracting key concepts from video...
Searching YouTube for: ['mobile rising system', 'thermal imaging camera', 'instant command control']
**Related Incident Training Videos**


 Key concept: Mobile Rising System
  - Online Incident Registration and Management system in [PHP ...
    https://www.youtube.com/watch?v=bqrHNBXOSf8
  - Project: RALLY! Training Troops & Vibing - YouTube
    https://www.youtube.com/watch?v=xM27Ne3kFHk
  - TRAINING and RANKING UP Players in FC Mobile (FIFA) - YouTube
    https://www.youtube.com/watch?v=fGpEg4Ivrrk

 Key concept: Thermal Imaging Camera
  - Incident Response with Drones & Thermal Imaging Cameras
    https://www.youtube.com/watch?v=S61CvBiHK3w
  - Thermal Imaging Camera Training for HVAC Techs - YouTube
    https://www.youtube.com/watch?v=4yn2Wf9s3oY
  - How to Use a Thermal Imaging Camera for Firefighting - YouTube
    https://www.youtube.com/watch?v=NWoIcM8uVIM

Based on key concepts automatically 

## 10. Collect all tools

We collect all six tools into a single list.
This list is passed directly to the LangGraph agent in Notebook 04.
The agent router uses the tool descriptions to decide which tool to call.

In [11]:
# All agent tools - passed directly to LangGraph in Notebook 04
AGENT_TOOLS = [
    fetch_youtube_transcript,      # Tool 1 - load video into ChromaDB
    search_video_knowledge,        # Tool 2 - answer questions via RAG
    summarize_video,               # Tool 3 - generate structured summary
    generate_pdf_cheatsheet,       # Tool 4 - create 1-page PDF
    send_gmail_with_cheatsheet,    # Tool 5 - send PDF to distribution list
    find_related_youtube_videos,   # Tool 6 - find related incident training videos
]

print('All tools collected.')
print(f'   Total tools available to agent: {len(AGENT_TOOLS)}')
print()
for i, t in enumerate(AGENT_TOOLS, 1):
    print(f'   Tool {i}: {t.name}')
    print(f'            {t.description[:80]}...')

All tools collected.
   Total tools available to agent: 6

   Tool 1: fetch_youtube_transcript
            Fetch the transcript of a YouTube video and store it in ChromaDB.
Use this tool ...
   Tool 2: search_video_knowledge
            Search the ChromaDB knowledge base for information relevant to the query.
Use th...
   Tool 3: summarize_video
            Generate a structured summary of the entire loaded video.
Use this tool when the...
   Tool 4: generate_pdf_cheatsheet
            Generate a professional 1-page PDF cheatsheet from the loaded video content.
Use...
   Tool 5: send_gmail_with_cheatsheet
            Send the generated PDF cheatsheet to the distribution list via Gmail.
Use this t...
   Tool 6: find_related_youtube_videos
            Find related YouTube incident training videos based on key concepts from the loa...


## 11. Tool quality tests

Automated tests to verify all tools work correctly before passing them to the LangGraph agent.
The Gmail tool is excluded from automated testing as it requires OAuth credentials.
The Tavily tool requires a valid API key - ensure TAVILY_API_KEY is set in .env.

In [12]:
print('=' * 60)
print('TOOL QUALITY TESTS')
print('=' * 60)

tests_passed = 0
tests_failed = 0

def check(name: str, condition: bool, detail: str = ''):
    global tests_passed, tests_failed
    if condition:
        tests_passed += 1
        print(f'  PASS - {name}')
    else:
        tests_failed += 1
        print(f'  FAIL - {name}')
    if detail:
        print(f'         {detail}')

# Test 1 - Tool 1 fetch returns confirmation
result_t1 = fetch_youtube_transcript.invoke({'youtube_url': 'https://www.youtube.com/watch?v=7OH5FEWWM_k'})
check('Tool 1 - fetch_youtube_transcript returns confirmation', 'Chunks stored' in result_t1, f'Result: {result_t1[:80]}...')

# Test 2 - Tool 1 handles invalid URL gracefully
result_invalid = fetch_youtube_transcript.invoke({'youtube_url': 'https://not-a-youtube-url.com'})
check('Tool 1 - handles invalid URL gracefully', 'Error' in result_invalid or 'Cannot' in result_invalid, f'Result: {result_invalid[:80]}')

# Test 3 - Tool 2 returns relevant context
result_t2 = search_video_knowledge.invoke({'query': 'What went wrong during the incident?'})
check('Tool 2 - search_video_knowledge returns relevant context', len(result_t2) > 100 and 'Sources' in result_t2, f'Result length: {len(result_t2)} characters')

# Test 4 - Tool 2 works with Dutch query
result_t2_nl = search_video_knowledge.invoke({'query': 'Welke lessen werden geleerd?'})
check('Tool 2 - handles Dutch query via automatic translation', len(result_t2_nl) > 100, f'Result length: {len(result_t2_nl)} characters')

# Test 5 - Tool 3 returns structured Dutch summary
result_t3 = summarize_video.invoke({'language': 'dutch'})
check('Tool 3 - summarize_video returns structured Dutch summary', len(result_t3) > 200 and ('**' in result_t3 or '-' in result_t3), f'Summary length: {len(result_t3)} characters')

# Test 6 - Tool 4 generates PDF file on disk
result_t4 = generate_pdf_cheatsheet.invoke({'language': 'dutch', 'source_url': 'https://www.youtube.com/watch?v=7OH5FEWWM_k'})
pdf_exists = False
if 'File path:' in result_t4:
    generated_path = result_t4.split('File path: ')[1].split('\n')[0]
    pdf_exists = Path(generated_path).exists()
check('Tool 4 - generate_pdf_cheatsheet creates PDF file on disk', pdf_exists, f'Result: {result_t4[:80]}...')

# Test 7 - Tool 5 is defined and callable
check('Tool 5 - send_gmail_with_cheatsheet is defined and callable', callable(send_gmail_with_cheatsheet), 'Gmail tool requires credentials.json for actual sending - tested during deployment')

# Test 8 - Tool 6 returns related videos
result_t6 = find_related_youtube_videos.invoke({'topic_hint': 'high rise fire'})
check('Tool 6 - find_related_youtube_videos returns video links', 'youtube.com' in result_t6 or 'Related' in result_t6, f'Result preview: {result_t6[:100]}...')

# Test 9 - Tool 6 works without topic hint
result_t6_auto = find_related_youtube_videos.invoke({'topic_hint': ''})
check('Tool 6 - auto-extracts concepts and finds videos without topic hint', len(result_t6_auto) > 100, f'Result length: {len(result_t6_auto)} characters')

# Test 10 - All 6 tools in AGENT_TOOLS list
check('All 6 tools collected in AGENT_TOOLS list', len(AGENT_TOOLS) == 6, f'Tools in list: {len(AGENT_TOOLS)}')

print('\n' + '=' * 60)
print(f'RESULTS: {tests_passed} passed | {tests_failed} failed')
if tests_failed == 0:
    print('All tests passed - tools are ready for Notebook 04!')
else:
    print('Fix the failing tests before moving to Notebook 04.')
print('=' * 60)

TOOL QUALITY TESTS
  PASS - Tool 1 - fetch_youtube_transcript returns confirmation
         Result: Transcript loaded successfully.
Video ID: 7OH5FEWWM_k
Total characters: 19,793
C...
  PASS - Tool 1 - handles invalid URL gracefully
         Result: Error fetching transcript: Cannot extract video ID from URL: https://not-a-youtu
  PASS - Tool 2 - search_video_knowledge returns relevant context
         Result length: 4052 characters
  PASS - Tool 2 - handles Dutch query via automatic translation
         Result length: 4045 characters
  PASS - Tool 3 - summarize_video returns structured Dutch summary
         Summary length: 1050 characters
Extracting keypoints for PDF...
Generating PDF...
  PASS - Tool 4 - generate_pdf_cheatsheet creates PDF file on disk
         Result: PDF cheatsheet generated successfully.
File path: /tmp/incidentiq_cheatsheet_202...
  PASS - Tool 5 - send_gmail_with_cheatsheet is defined and callable
         Gmail tool requires credentials.json for actual sending

---

## What we built in this notebook

| Tool | Name | Trigger | What it does |
|------|------|---------|-------------|
| 1 | fetch_youtube_transcript | YouTube URL detected | Fetches transcript and stores in ChromaDB |
| 2 | search_video_knowledge | Question about video | RAG retrieval with auto-translation |
| 3 | summarize_video | User asks for summary | Structured summary in any language |
| 4 | generate_pdf_cheatsheet | User asks for cheatsheet | Branded 1-page PDF with AI disclaimer |
| 5 | send_gmail_with_cheatsheet | User asks to send email | PDF to distribution list plus custom email |
| 6 | find_related_youtube_videos | User asks for more resources | Related incident training videos via Tavily |

## Who can use these tools
All tools use incident training terminology - not service specific.
Any emergency or security organization can use IncidentIQ out of the box:
fire services, police forces, EMS teams, civil protection and military training.

